## Persistence

In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [2]:

load_dotenv()

llm = ChatOpenAI()

In [3]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [4]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [5]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [6]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [7]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why did the pizza go to the doctor? Because it was feeling a little saucy!',
 'explanation': 'This joke is a play on words, combining the literal meaning of "sauce" as a topping on pizza with the slang definition of being "saucy" meaning sassy or cheeky. In this joke, the pizza is humorously anthropomorphized as having feelings and needing to see a doctor because it is feeling sassy or spirited.'}

In [8]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the doctor? Because it was feeling a little saucy!', 'explanation': 'This joke is a play on words, combining the literal meaning of "sauce" as a topping on pizza with the slang definition of being "saucy" meaning sassy or cheeky. In this joke, the pizza is humorously anthropomorphized as having feelings and needing to see a doctor because it is feeling sassy or spirited.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f121127-b8b3-6ac6-8002-e585888d4889'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-03-16T08:31:09.104599+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f121127-a495-6a3b-8001-4eac018825b6'}}, tasks=(), interrupts=())

In [9]:

list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the doctor? Because it was feeling a little saucy!', 'explanation': 'This joke is a play on words, combining the literal meaning of "sauce" as a topping on pizza with the slang definition of being "saucy" meaning sassy or cheeky. In this joke, the pizza is humorously anthropomorphized as having feelings and needing to see a doctor because it is feeling sassy or spirited.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f121127-b8b3-6ac6-8002-e585888d4889'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-03-16T08:31:09.104599+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f121127-a495-6a3b-8001-4eac018825b6'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the doctor? Because it was feeling a little saucy!'}, next=('generate_explanation',),

In [10]:

config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': 'Why did the spaghetti go to the party? \nBecause it heard it was gonna be a pasta-bilities!',
 'explanation': 'This joke plays on the words "party" and "pasta-bilities." The spaghetti went to the party because it heard that there were going to be a lot of pasta dishes available, so it saw it as a great opportunity to enjoy some delicious food. The pun on "possibilities" and "pasta-bilities" adds a playful twist to the joke, making it a light-hearted and humorous play on words.'}

In [11]:

workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the doctor? Because it was feeling a little saucy!', 'explanation': 'This joke is a play on words, combining the literal meaning of "sauce" as a topping on pizza with the slang definition of being "saucy" meaning sassy or cheeky. In this joke, the pizza is humorously anthropomorphized as having feelings and needing to see a doctor because it is feeling sassy or spirited.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f121127-b8b3-6ac6-8002-e585888d4889'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-03-16T08:31:09.104599+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f121127-a495-6a3b-8001-4eac018825b6'}}, tasks=(), interrupts=())

## Time Travel

In [14]:

workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f121127-a495-6a3b-8001-4eac018825b6"}})

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the doctor? Because it was feeling a little saucy!'}, next=('generate_explanation',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f121127-a495-6a3b-8001-4eac018825b6'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-03-16T08:31:06.995145+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f121127-925b-6704-8000-65da53b9f2e6'}}, tasks=(PregelTask(id='d3a8f6d3-eb1f-b48a-9a06-87cea617633d', name='generate_explanation', path=('__pregel_pull', 'generate_explanation'), error=None, interrupts=(), state=None, result={'explanation': 'This joke is a play on words, combining the literal meaning of "sauce" as a topping on pizza with the slang definition of being "saucy" meaning sassy or cheeky. In this joke, the pizza is humorously anthropomorphized as having feelings and needing to see a doctor because it is feeling sassy or spirited.'}

In [18]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f121127-a495-6a3b-8001-4eac018825b6"}})

{'topic': 'pizza',
 'joke': 'Why did the pizza go to the doctor? Because it was feeling a little saucy!',
 'explanation': 'This joke plays on the word "saucy," which can mean either having a bold or impudent attitude, or containing a lot of sauce. In this case, the joke suggests that the pizza went to the doctor because it was feeling bold or impudent, which is humorous because pizzas are typically not sentient beings capable of feeling emotions. The double meaning of "saucy" adds an extra layer of humor to the punchline.'}

In [19]:

list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the doctor? Because it was feeling a little saucy!', 'explanation': 'This joke plays on the word "saucy," which can mean either having a bold or impudent attitude, or containing a lot of sauce. In this case, the joke suggests that the pizza went to the doctor because it was feeling bold or impudent, which is humorous because pizzas are typically not sentient beings capable of feeling emotions. The double meaning of "saucy" adds an extra layer of humor to the punchline.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f12112e-080a-6d78-8002-714f041b64b9'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-03-16T08:33:58.485303+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f121127-a495-6a3b-8001-4eac018825b6'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did t

## Updating State

In [20]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f121127-a495-6a3b-8001-4eac018825b6", "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f12112f-7917-6090-8002-60efbae75bc6'}}

In [21]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa', 'joke': 'Why did the pizza go to the doctor? Because it was feeling a little saucy!'}, next=('generate_explanation',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f12112f-7917-6090-8002-60efbae75bc6'}}, metadata={'source': 'update', 'step': 2, 'parents': {}}, created_at='2026-03-16T08:34:37.182776+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f121127-a495-6a3b-8001-4eac018825b6'}}, tasks=(PregelTask(id='33b3f7f9-a97a-9da5-9b25-a45436b979c7', name='generate_explanation', path=('__pregel_pull', 'generate_explanation'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the doctor? Because it was feeling a little saucy!', 'explanation': 'This joke plays on the word "saucy," which can mean either having a bold or impudent attitude, or containing a lot of sauce. In th

In [22]:

workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f12112f-7917-6090-8002-60efbae75bc6"}})

{'topic': 'samosa',
 'joke': 'Why did the pizza go to the doctor? Because it was feeling a little saucy!',
 'explanation': 'This joke plays on the double meaning of the word "saucy." In one context, "saucy" can mean cheeky or impudent, implying that the pizza was being mischievous by going to the doctor. In another context, "saucy" can refer to having a lot of sauce, which could be a symptom of feeling unwell or needing medical attention. The punchline combines both meanings cleverly to create a humorous twist.'}